In [1]:
from tokenizers import Tokenizer
from tokenizers.processors import TemplateProcessing
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import pandas as pd
import numpy as np
import copy
from tqdm import tqdm

In [2]:
train_df = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv').dropna()
test_df = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv')

In [3]:
tokenizer = Tokenizer.from_file('/kaggle/input/notebooks/naneet1/blockformer-tokenizer/BlockFormer.json')

bos_id = tokenizer.token_to_id("<bos>")
eos_id = tokenizer.token_to_id("<eos>")

tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[
        ("<bos>", bos_id),
        ("<eos>", eos_id),
    ],
)

In [4]:
class TinyStories(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df['text'].iloc[idx]
        tokens = self.tokenizer.encode(text).ids
        input_ids = torch.tensor(tokens[:-1])
        target = torch.tensor(tokens[1:])
        return input_ids, target

In [5]:
def collate_fn(batch):
    
    input_ids, target = zip(*batch)
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    target = pad_sequence(target, batch_first=True, padding_value=0)

    b, s = input_ids.shape

    attention_mask = (input_ids == 0)

    causal_mask = torch.triu(
        torch.ones(s, s), diagonal=1
    ).bool()

    return input_ids, attention_mask, causal_mask, target

In [6]:
train_dataset = TinyStories(df=train_df, tokenizer=tokenizer)
test_dataset = TinyStories(df=test_df, tokenizer=tokenizer)

In [7]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    prefetch_factor=10
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    prefetch_factor=8
)

In [8]:
input_ids, attention_mask, causal_mask, target = next(iter(train_dataloader))

In [9]:
input_ids.shape, attention_mask.shape, causal_mask.shape, target.shape

(torch.Size([16, 316]),
 torch.Size([16, 316]),
 torch.Size([316, 316]),
 torch.Size([16, 316]))

In [10]:
input_ids, target, attention_mask, causal_mask

(tensor([[   1,  335,  258,  ...,    0,    0,    0],
         [   1, 3061,  300,  ...,    0,    0,    0],
         [   1, 3061,  350,  ...,    0,    0,    0],
         ...,
         [   1,  250,  163,  ...,  164, 2523,   17],
         [   1,  365,  163,  ...,    0,    0,    0],
         [   1, 3061,  350,  ...,    0,    0,    0]]),
 tensor([[ 335,  258,   15,  ...,    0,    0,    0],
         [3061,  300,  183,  ...,    0,    0,    0],
         [3061,  350,  157,  ...,    0,    0,    0],
         ...,
         [ 250,  163,  365,  ..., 2523,   17,    2],
         [ 365,  163,  685,  ...,    0,    0,    0],
         [3061,  350,  157,  ...,    0,    0,    0]]),
 tensor([[False, False, False,  ...,  True,  True,  True],
         [False, False, False,  ...,  True,  True,  True],
         [False, False, False,  ...,  True,  True,  True],
         ...,
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ...,  True,  True,  True],
         [False, False,

In [11]:
scaler = GradScaler('cuda')

def train_step(model, optimizer, dataloader, epoch, device="cuda"):
    model.train()
    train_loss = 0

    # progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}", dynamic_ncols=True)
    
    for batch_idx, (input_ids, attention_mask, causal_mask, target) in enumerate(dataloader):
        input_ids, attention_mask, causal_mask, target = input_ids.to(device), attention_mask.to(device), causal_mask.to(device), target.to(device)

        with autocast('cuda'):
            logits = model(input_ids, attention_mask, causal_mask)
            loss = F.cross_entropy(logits.permute(0,2,1), target, ignore_index=0, label_smoothing=0.1)

            batch_loss = loss.item()
            train_loss += batch_loss

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        # avg_loss = train_loss / (batch_idx + 1)
        # progress_bar.set_postfix(batch_loss=f"{batch_loss:.4f}", avg_loss=f"{avg_loss:.4f}")

        del input_ids, attention_mask, causal_mask, loss, logits

    avg_loss = train_loss / len(dataloader)

    print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f}")
    return avg_loss



def test_step(model, epoch, dataloader, device="cuda"):
    model.eval()
    test_loss, total_tokens = 0, 0

    # progress_bar = tqdm(dataloader, desc=f"Epoch {epoch} (Eval)", dynamic_ncols=True)

    with torch.no_grad():
        for batch_idx, (input_ids, attention_mask, causal_mask, target) in enumerate(dataloader):
            input_ids, attention_mask, causal_mask, target = input_ids.to(device), attention_mask.to(device), causal_mask.to(device), target.to(device)

            with autocast('cuda'):
                logits = model(input_ids, attention_mask, causal_mask)
                loss = F.cross_entropy(logits.permute(0,2,1), target, ignore_index=0, label_smoothing=0.1, reduction='sum')
    
                batch_loss = loss.item()
                test_loss += batch_loss
                batch_tokens = (~attention_mask).sum().item()
                total_tokens += batch_tokens

            # batch_ppl = torch.exp(torch.tensor(batch_loss / batch_tokens))
            # progress_bar.set_postfix(batch_ppl=f"{batch_ppl:.4f}")
    
            del input_ids, attention_mask, causal_mask, loss, logits

    avg_loss = test_loss / total_tokens
    avg_nll = test_loss / total_tokens
    ppl = torch.exp(torch.tensor(avg_nll))

    print(f"Epoch {epoch} | Test Loss: {avg_loss:.4f} | Test Preplexity: {ppl:.4f}")
    return avg_loss, ppl

In [ ]:
class RoPE(nn.Module):
    def __init__(
        self,
        nhead,
        d_model,
        max_seq_len
    ):
        super().__init__()
        head_dim = d_model // nhead
        assert head_dim % 2 == 0
        d = head_dim

        k = torch.arange(head_dim // 2, dtype=torch.float32)
        inv_freq = 10000 ** (-2 * k / head_dim)

        positions = torch.arange(max_seq_len, dtype=torch.float32)
        angles = torch.outer(positions, inv_freq)

        cos_cache = torch.cos(angles).repeat_interleave(repeats=2, dim=-1)[None, None, :, :]
        sin_cache = torch.sin(angles).repeat_interleave(repeats=2, dim=-1)[None, None, :, :]

        self.register_buffer('cos_cache', cos_cache)
        self.register_buffer('sin_cache', sin_cache)


    def rotate(self, x):
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        rotate_even = -x_odd
        rotate_odd = x_even

        rotate = torch.stack([rotate_even, rotate_odd], dim=-1).reshape(*x.shape)

        return rotate

    def forward(self, x, offset=None):
        B, H, S, D = x.shape

        if offset == None:
            cos = self.cos_cache[:,:,:S,:]
            sin = self.sin_cache[:,:,:S,:]

        else:
            cos = self.cos_cache[:,:, offset:offset+S,:]
            sin = self.sin_cache[:,:, offset:offset+S,:]

        x = x * cos + self.rotate(x) * sin

        return x

In [ ]:
class MHA(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        rope,
        bias=True,
        dropout=0.0,
    ):
        super().__init__()

        assert d_model % nhead == 0
        self.nhead = nhead
        self.d_model = d_model
        self.head_dim = d_model // nhead
        assert self.head_dim % 2 == 0

        self.rope = rope

        self.w_q = nn.Linear(d_model, d_model, bias=bias)
        self.w_k = nn.Linear(d_model, d_model, bias=bias)
        self.w_v = nn.Linear(d_model, d_model, bias=bias)
        self.w_o = nn.Linear(d_model, d_model, bias=bias)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        input_ids,
        causal_mask=None,
        attention_mask=None,
        kv_cache=None,
        return_cache=False
    ):
        
        B, S, E = input_ids.shape
        query = self.w_q(input_ids).view(B, S, self.nhead, self.head_dim).permute(0, 2, 1, 3)
        key = self.w_k(input_ids).view(B, S, self.nhead, self.head_dim).permute(0, 2, 1, 3)
        value = self.w_v(input_ids).view(B, S, self.nhead, self.head_dim).permute(0, 2, 1, 3)

        if kv_cache != None:
            offset = kv_cache[0].shape[2]
            
            key = self.rope(key, offset=offset)
            key = torch.cat([kv_cache[0], key], dim=2)
            
            value = torch.cat([kv_cache[1], value], dim=2)
            
            kv_cache = [key, value]

            query = self.rope(query, offset=offset)

        elif return_cache:
            key = self.rope(key, offset=None)
            kv_cache = [key, value]
            query = self.rope(query, offset=None)

        else:
            query = self.rope(query)
            key = self.rope(key)

        attention = query @ key.transpose(-1, -2)
        attention = attention / (self.head_dim ** 0.5)

        if causal_mask is not None:
            causal_mask = causal_mask[None, None, :, :]
            attention = attention.masked_fill(causal_mask, float('-inf'))

        if attention_mask is not None:
            attention_mask = attention_mask[:, None, None, :]
            attention = attention.masked_fill(attention_mask, float('-inf'))

        attention = torch.softmax(attention, dim=-1)
        attention = self.dropout(attention)

        output = attention @ value
        output = output.transpose(1, 2).contiguous().view(B, S, self.d_model)
        output = self.w_o(output)
        output = self.dropout(output)

        if kv_cache == None and not return_cache:
            return output

        elif kv_cache != None or return_cache:
            return output, kv_cache

In [ ]:
class TransformerDecoderLayer(nn.Module):
    def __init__(
        self,
        rope,
        d_model=512,
        nhead=8,
        dim_ff=2048,
        activation=nn.GELU(),
        bias=True,
        dropout=0.0
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.mha = MHA(
            d_model=d_model,
            nhead=nhead,
            rope=rope,
            bias=bias,
            dropout=dropout,
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.dff = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            activation,
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        causal_mask,
        kv_cache=None,
        return_cache=False
    ):
        
        x = input_ids
        residual = x

        if kv_cache != None or return_cache:
            x, kv_cache = self.mha(
                input_ids=self.norm1(x),
                attention_mask=attention_mask,
                causal_mask=causal_mask,
                kv_cache=kv_cache,
                return_cache=return_cache
            )
        else:
            x = self.mha(
                input_ids=self.norm1(x),
                attention_mask=attention_mask,
                causal_mask=causal_mask,
                kv_cache=kv_cache,
                return_cache=return_cache
            )
            
        x = residual + x
        
        residual = x
        x = self.dff(self.norm2(x))
        x = residual + x

        if kv_cache != None or return_cache:
            return x, kv_cache
        else:
            return x


In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(
        self,
        decoderlayer,
        num_layers
    ):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(decoderlayer) for _ in range(num_layers)])

    def forward(
        self,
        input_ids,
        attention_mask,
        causal_mask,
        kv_cache=None,
        return_cache=False
    ):
        if return_cache and kv_cache == None:
            use_cache = True
            kv_cache = [None for _ in range(len(self.layers))]
        elif kv_cache is not None:
            use_cache = True
        else:
            use_cache = False
        x = input_ids

        for n, decoderlayer in enumerate(self.layers):
            if use_cache:
                x, kv_cache[n] = decoderlayer(
                    input_ids=x,
                    attention_mask=attention_mask,
                    causal_mask=causal_mask,
                    kv_cache=kv_cache[n],
                    return_cache=return_cache
                )

            elif return_cache:
                x, kv_cache[n] = decoderlayer(
                    input_ids=x,
                    attention_mask=attention_mask,
                    causal_mask=causal_mask,
                    kv_cache=None,
                    return_cache=return_cache
                )

            else:
                x = decoderlayer(
                    input_ids=x,
                    attention_mask=attention_mask,
                    causal_mask=causal_mask,
                    kv_cache=None,
                    return_cache=return_cache
                )

        if kv_cache != None or return_cache:
            return x, kv_cache
        else:
            return x

In [ ]:
class BlockFormer(nn.Module):
    def __init__(
        self,
        d_model=512,
        nhead=8,
        num_layers=6,
        dim_ff=2048,
        dropout=0.15,
        vocab=16000,
        max_deq_len=2048
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab,
            embedding_dim=d_model,
            padding_idx=0
        )

        rope = RoPE(nhead=nhead,
                   d_model=d_model,
                   max_seq_len=max_deq_len)

        self.decoder = TransformerDecoder(
            TransformerDecoderLayer(
                rope=rope,
                d_model=512,
                nhead=8,
                dim_ff=2048,
                activation=nn.GELU(),
                bias=True,
                dropout=dropout
            ),
            num_layers=num_layers
        )

        self.ln = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab)
        self.fc.weight = self.embedding.weight

    def forward(
        self, 
        input_ids, 
        attention_mask=None, 
        causal_mask=None, 
        kv_cache=None, 
        return_cache=False
    ):

        b, s = input_ids.shape
    
        causal_mask = torch.triu(
            torch.ones(s, s), diagonal=1
        ).bool().to('cuda')
        
        x = self.embedding(input_ids)

        if kv_cache != None or return_cache:
            x, kv_cache = self.decoder(
                input_ids=x,
                causal_mask=causal_mask,
                attention_mask=attention_mask,
                kv_cache=kv_cache,
                return_cache=return_cache
            )

            x = self.ln(x)
            logits = self.fc(x)
            return logits, kv_cache

        else:
            x = self.decoder(
                input_ids=x,
                causal_mask=causal_mask,
                attention_mask=attention_mask,
                kv_cache=kv_cache,
                return_cache=return_cache
            )

        x = self.ln(x)
        logits = self.fc(x)

        if kv_cache != None or return_cache:
            return logits, kv_cache
            
        else:
            return logits

    @torch.inference_mode()
    def generate(
        self, 
        input_ids, 
        max_new_tokens, 
        use_cache=True
    ):

        max_len = input_ids.size(1) + max_new_tokens
        full_mask = torch.triu(
            torch.ones(max_len, max_len), diagonal=1
        ).bool().to(input_ids.device)
        
        b, s = input_ids.shape
        preds = []
        
        for _ in range(max_new_tokens):
            if _ == 0 and use_cache:
                kv_cache = None
                return_cache=True
                preds = [input_ids]
                
            elif _ == 0 and not use_cache:
                kv_cache = None
                return_cache=False
            
            causal_mask = full_mask[:s, :s]

            if use_cache:
                logits, kv_cache = self.forward(
                    input_ids=input_ids,
                    attention_mask=None,
                    causal_mask=causal_mask,
                    kv_cache=kv_cache,
                    return_cache=return_cache
                )

            else:
                logits = self.forward(
                    input_ids=input_ids,
                    attention_mask=None,
                    causal_mask=causal_mask
                )

            temperature = 1
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

            if use_cache:
                preds.append(next_token)
                input_ids = next_token

            else:
                input_ids = torch.cat([input_ids, next_token], dim=1)

            s+=1
        if use_cache:
            preds = torch.cat(preds, dim=1)
            return preds

        else:
            return input_ids

In [17]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model = BlockFormer(
    d_model=512,
    nhead=8,
    num_layers=6,
    dim_ff=2048,
    dropout=0.15,
    vocab=16000,
    max_deq_len=2048
).to('cuda')

model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01, betas=(0.9, 0.95))

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,   
    eta_min=3e-5       
)

checkpoint = torch.load("/kaggle/input/notebooks/oliveseed/blockformer-trainer/BlockFormer_13_epochs.pth", map_location="cuda")

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

print("Checkpoint loaded successfully.")

Checkpoint loaded successfully.


In [18]:
current = 13
num_epochs = 1

start = 1 + current
epochs = num_epochs + current + 1

print(f"Running from {start} to {epochs-1}")

torch.manual_seed(42)
torch.cuda.manual_seed(42)

print("Ready to train!!")

for epoch in range(start, epochs):
    train_step(
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        dataloader=train_dataloader
    )

    test_step(
        model=model,
        epoch=epoch,
        dataloader=test_dataloader,
    )
    # scheduler.step()
    checkpoint = {
        'model_state_dict' : model.state_dict(),
        'optimizer_state_dict' : optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict()
    }
    torch.save(obj=checkpoint, f=f"BlockFormer_{epoch}_epochs.pth")

Running from 14 to 14
Ready to train!!
Epoch 14 | Train Loss: 3.2483
Epoch 14 | Test Loss: 3.1034 | Test Preplexity: 22.2732
